In [ ]:
# Run this code to install the missing packages
!pip install qiskit
!pip install qiskit_aer
!pip install pylatexenc

# Simon's algorithm

We implement Simon's algorithm on a custom periodic function. The function $f$ has period $a=010$.

In [ ]:
# Simon Algorithm in Qiskit

from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit_aer import QasmSimulator
from qiskit.visualization import plot_histogram
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt

# Function defining a 2-to-1 mapping with hidden period a = 010
f_map = {
    "000": "000",
    "010": "000",  # 000 xor 010 = 010
    "001": "001",
    "011": "001",  # 001 xor 010 = 011
    "100": "010", 
    "110": "010",  # 100 xor 010 = 110
    "101": "011",
    "111": "011"   # 101 xor 010 = 111
}

We can map this function inside a unitary operator by iteratively creating the unitary matrix and use the class `UnitaryGate` offered in `qiskit`.

In [ ]:

# Build the oracle Uf: |x>|y> --> |x>|y xor f(x)|
def build_simon_oracle(f_map, n_qubits):
    N = 2 ** n_qubits
    Uf_matrix = np.eye(N)

    # Iteratively build the unitary
    for i in range(N):
        # Split i in 2n-bit string → x = input (leftmost n), y = output (rightmost n)
        i_bin = format(i, f'0{n_qubits}b')  # stringa a 2n bit
        x = i_bin[:(n_qubits // 2)]
        y = i_bin[(n_qubits // 2):]
        # Data x: n/2 leftmost bits  
        # Computation y: n/2 rightmost bits
        fx = f_map[x]
        y_out = format(int(y, 2) ^ int(fx, 2), f'0{n_qubits // 2}b')
        
        j_bin = x + y_out
        j = int(j_bin, 2)
        #j = (int(x, 2) << 2) | int(y_out, 2)
        Uf_matrix[i][i] = 0
        Uf_matrix[i][j] = 1

    Uf_gate = UnitaryGate(Uf_matrix, label="Uf")
    return Uf_gate

# Build the full Simon circuit
def build_simon_circuit(Uf_gate, n_qubits):
    qc = QuantumCircuit(n_qubits, n_qubits // 2)  # 2 input qubits, 2 output, measure only inputs

    # Step 1: put input in uniform superposition
    qc.h(range(n_qubits // 2))

    # Step 2: apply oracle Uf
    # Order of qubits is inverted to match the ordering of qubits in Qiskit
    qc.append(Uf_gate, reversed(range(n_qubits)))

    # Step 3: discard output, apply Hadamard again to inputs
    qc.h(range(n_qubits // 2))

    # Step 4: measure inputs
    qc.measure(range(n_qubits // 2), range(n_qubits // 2))

    return qc

n_qubits = 2 * int(np.log2(len(f_map.keys())))
# Build oracle and circuit
Uf_gate = build_simon_oracle(f_map, n_qubits)
qc_simon = build_simon_circuit(Uf_gate, n_qubits)
qc_simon.draw('mpl')


Let's simualte the circuit and get some measurements:

In [ ]:
backend = QasmSimulator(shots=10)

job = backend.run(qc_simon)
result = job.result()
counts = result.get_counts()

# Mostra istogramma dei risultati
plot_histogram(counts)

We observe 4 different states! But we need only 3 of them to find the period $a$. Let's solve:
$$
\begin{cases}
& (1 \cdot a_0 +  0 \cdot a_1 + 0 \cdot a_2) \mod 2 = 0 \\
& (0 \cdot a_0 +  0 \cdot a_1 + 1 \cdot a_2) \mod 2 = 0 \\
& (1 \cdot a_0 +  0 \cdot a_1 + 1 \cdot a_2) \mod 2 = 0 \\
\end{cases}
$$
from which we derive $a=010$.